# روایت اجرایی v14 → v16b — زنجیرهٔ کارخانهٔ واژگان (۵۰۰ لم پایلوت)

همهٔ اعداد این دفتر از **اجرای کد روی فایل‌های فیکسچر** در همین اجرا به دست آمده‌اند؛ هیچ عددی دستی یا حدسی نیست. سلول‌های کد فیکسچرها را با مسیر نسبی `../fixtures/...` می‌خوانند، پس دفتر هرجا که فیکسچرها باشند دوباره اجرا می‌شود. بدون تماس API، بدون خواندن `factory/.env`، بدون چاپ کلید.

Persian narrative in markdown cells; English code and outputs. Small embedded data only (counts, probe rows, distributions).

## ۱) نقشهٔ زنجیره — v14a ← v14b ← v14c ← v15 ← v16 ← v16b

- **v14a** قطعی، صفر تماس: وزن‌های جدید + نگهبان EVP + ادغام عین‌هم (۱۵ مورد با `merged_from`).
- **v14b** ادغام مدل (paraphrase merge): کارت‌ها ۲۱۰۱ ← ۱۶۷۶.
- **v14c** داور (judge) + موضوع: انتخاب ۲/۳/۴ تایی هر سطح + برچسب تک‌موضوعی.
- **v15** بردار وزن‌دار موضوع (تا ۳ برچسب، جمع ۱٫۰) بدون تصمیم مجدد — همان برچسب‌های v14c.
- **v16** بازبرچسب تازه با ۱۶ سرموضوع (۳ شکافتن: Nature→دو، Work&Education→دو، Society&Culture→دو).
- **v16b** تکمیل (top-up): فقط ۵۵۲ حس Otherِ v16 با پرامپت قوی‌تر دوباره برچسب خورد.

جدول زیر را سلول بعد از روی فایل‌ها چاپ می‌کند (تعداد تماس‌ها: بچ‌های progress + اعداد ثبت‌شده در evidenceها).

In [1]:
import json, re, sqlite3, collections
from pathlib import Path

FIX = Path('../fixtures')
FAC = Path('..')
assert FIX.is_dir(), FIX.resolve()

def load(name):
    return json.loads((FIX / name).read_text(encoding='utf-8'))

ranked_a = load('ranked_senses-v14a.json')
ranked_b = load('ranked_senses-v14b.json')
ranked_c = load('ranked_senses-v14c.json')
uniq_a = load('uniq_senses-v14a.json')
lab_a = load('topic_labels-v14a.json')
lab_c = load('topic_labels-v14c.json')
lab_16 = load('topic_labels-v16.json')
lab_16b = load('topic_labels-v16b.json')
vec_15 = load('topic_vectors-v15.json')
vec_16 = load('topic_vectors-v16.json')
vec_16b = load('topic_vectors-v16b.json')
pack = json.loads((FAC / 'packs/en/pack.json').read_text(encoding='utf-8'))
print('loaded: ranked', len(ranked_a), len(ranked_b), len(ranked_c), '| uniq entries', len(uniq_a))
print('labels:', len(lab_a), len(lab_c), len(lab_16), len(lab_16b))
def cards(ranked): return sum(len(e['ranked_senses']) for e in ranked)
ca, cb, cc = cards(ranked_a), cards(ranked_b), cards(ranked_c)
n_uniq = sum(len(e['uniq_senses']) for e in uniq_a)
prog = {}
for k, f in [('merge', 'v14_merge_progress.json'), ('judge', 'v14_judge_progress.json'),
             ('v15', 'v15_progress.json'), ('v16', 'v16_progress.json'), ('v16b', 'v16b_progress.json')]:
    d = json.loads((FAC / f).read_text(encoding='utf-8'))
    prog[k] = (d['done_batches'], d['total_batches'])
# calls ledger: merge/judge = 1 call per batch; v15/v16/v16b = batches + documented smoke calls (see evidence files, section 6 re-derives)
calls = {'v14a': 0, 'v14b': prog['merge'][0], 'v14c': prog['judge'][0], 'v15': 64, 'v16': 64, 'v16b': 39}
print(f"{'version':8} {'lemmas':>7} {'cards':>7} {'calls':>6}  what")
for v, n, c0, w in [('v14a', len(ranked_a), ca, 'deterministic weights/recall/exact-merge'),
                    ('v14b', len(ranked_b), cb, 'LLM paraphrase merge'),
                    ('v14c', len(ranked_c), cc, 'judge picks + topics'),
                    ('v15', 500, cc, 'weight vectors, no re-decisions'),
                    ('v16', 500, cc, 'fresh relabel, 13 -> 16 heads'),
                    ('v16b', 500, cc, 'top-up on v16 Others only')]:
    print(f"{v:8} {n:>7} {c0:>7} {calls[v]:>6}  {w}")
print('uniq senses total:', n_uniq, '| progress batches:', prog)
assert (len(ranked_a), len(ranked_b), len(ranked_c)) == (500, 500, 500)
assert (ca, cb, cc) == (2101, 1676, 1676)
assert n_uniq == 4093
assert prog == {'merge': (63, 63), 'judge': (63, 63), 'v15': (63, 63), 'v16': (63, 63), 'v16b': (38, 38)}
print('CHAIN ASSERTS PASS: 500 lemmas, 2101 -> 1676 cards, 4093 uniq')

loaded: ranked 500 500 500 | uniq entries 500
labels: 2101 1676 1676 1676
version   lemmas   cards  calls  what
v14a         500    2101      0  deterministic weights/recall/exact-merge
v14b         500    1676     63  LLM paraphrase merge
v14c         500    1676     63  judge picks + topics
v15          500    1676     64  weight vectors, no re-decisions
v16          500    1676     64  fresh relabel, 13 -> 16 heads
v16b         500    1676     39  top-up on v16 Others only
uniq senses total: 4093 | progress batches: {'merge': (63, 63), 'judge': (63, 63), 'v15': (63, 63), 'v16': (63, 63), 'v16b': (38, 38)}
CHAIN ASSERTS PASS: 500 lemmas, 2101 -> 1676 cards, 4093 uniq


## ۲) تغییر فرمول وزن (R1)

قرارداد قفل R1 وزن قدیم v13 (فرکانس خام ۰٫۵۰، CEFR ‏۰٫۱۲، تشابه ۰٫۳۰) را کنار گذاشت، چون سلطهٔ Zipf و غیبت stone قابل اصلاح نبود. فرمول جدید `factory/packs/en/pack.json` شش مؤلفه دارد: فرکانس per-sense ‏۰٫۳۰ + CEFR نامتقارن ۰٫۳۵ + شمار WordNet ‏۰٫۱۰ + مرکزیت ۰٫۰۵ + موضوع ۰٫۱۰ + برخورد Tatoeba ‏۰٫۱۰، با گیت‌های P_REGISTER و P_POS. سلول بعد دقیقاً همین فایل را می‌خواند و assert می‌کند.

In [2]:
w = pack['weights']
print('pack weights from factory/packs/en/pack.json:', w)
assert abs(w['w_freq'] - 0.30) < 1e-9
assert abs(w['w_cefr'] - 0.35) < 1e-9
assert abs(w['w_wn'] - 0.10) < 1e-9
assert abs(w['w_cent'] - 0.05) < 1e-9
assert abs(w['w_topic'] - 0.10) < 1e-9
assert abs(w['w_tatoeba'] - 0.10) < 1e-9
assert abs(sum(w.values()) - 1.0) < 1e-9
# v13 recorded formula (CONTEXT-v14: keyword-831/thr-0.50 not reproducible, 8/400 sim>=0.50) vs v14 locked R1
print(f"{'component':10} {'v13 (recorded)':>15} {'v14 (pack.json)':>16}")
for k, old in [('freq', 0.50), ('cefr', 0.12), ('wn/sim', 0.30)]:
    new = {'freq': w['w_freq'], 'cefr': w['w_cefr'], 'wn/sim': w['w_wn']}[k]
    print(f"{k:10} {old:>15.2f} {new:>16.2f}")
print('+ v14 new: centroid 0.05, topic 0.10, tatoeba-hit 0.10; gates P_REGISTER 0.5/0.6/0.8, P_POS 0.7 (contract R1)')
print('WEIGHT ASSERTS PASS')

pack weights from factory/packs/en/pack.json: {'w_freq': 0.3, 'w_cefr': 0.35, 'w_wn': 0.1, 'w_cent': 0.05, 'w_topic': 0.1, 'w_tatoeba': 0.1}
component   v13 (recorded)  v14 (pack.json)
freq                  0.50             0.30
cefr                  0.12             0.35
wn/sim                0.30             0.10
+ v14 new: centroid 0.05, topic 0.10, tatoeba-hit 0.10; gates P_REGISTER 0.5/0.6/0.8, P_POS 0.7 (contract R1)
WEIGHT ASSERTS PASS


## ۳) پروب‌ها — rock / light / pass / flat / supporter / communicate / time

برای هر لم: حس top-1 (اول رتبه‌بندی v14c) + انتخاب‌های هر سطح (picks داور) + مسیر موضوع از v14a تا v16b. نکته‌های از پیش ثبت‌شده در CONTEXT: مبتدی rock یعنی #۲۳ «تکان» (موسیقی فقط پیشرفته)، top-1 رسمی flat یعنی آپارتمان، light با fallback قطعی [#۶۲ لامپ، #۷ سبک]، و مبتدی pass یعنی [#۸ گذراندن، #۲۵ بلیط]. «امتحان» در داده نیست.

In [3]:
by_lemma_c = {e['lemma']: e for e in ranked_c}
lab_by_ver = {'v14a': {r['sense_id']: r['topic_label'] for r in lab_a},
              'v14c': {r['sense_id']: r['topic_label'] for r in lab_c},
              'v16': {r['sense_id']: r['topic_label'] for r in lab_16},
              'v16b': {r['sense_id']: r['topic_label'] for r in lab_16b}}
vec16b = {v['sense_id']: v['vector'] for e in vec_16b for v in e['vectors']}
for lemma in ['rock', 'light', 'pass', 'flat', 'supporter', 'communicate', 'time']:
    e = by_lemma_c[lemma]
    top = e['ranked_senses'][0]
    print(f"=== {lemma} (tier {e['tier']}, pick_source {e['pick_source']})")
    print(f"  top-1: {top['sense_id']} | {top['gloss'][:70]} | v14c-topic: {top.get('topic_label')}")
    print(f"  picks: beginner={e['picks']['beginner']} intermediate={e['picks']['intermediate']} advanced={e['picks']['advanced']}")
    traj = ' -> '.join(f"{v}:{lab_by_ver[v].get(top['sense_id'], '?')}" for v in ['v14a', 'v14c', 'v16', 'v16b'])
    print(f"  top-1 topic trajectory: {traj}")
    print(f"  v16b vector: {[(x['topic_label'], x['weight']) for x in vec16b[top['sense_id']]]}")
# targeted asserts from locked CONTEXT findings
assert by_lemma_c['flat']['ranked_senses'][0]['sense_id'] == 'flat#14'  # top-1 apartment
assert by_lemma_c['rock']['picks']['beginner'] == ['rock#23', 'rock#4']  # beginner sway, music advanced-only
assert by_lemma_c['light']['pick_source'] == 'deterministic'
assert by_lemma_c['light']['picks']['beginner'] == ['light#62', 'light#7']
assert by_lemma_c['pass']['picks']['beginner'] == ['pass#8', 'pass#25']
assert not any(re.search(r'\\bexam\\b', s['gloss'], re.I) for s in by_lemma_c['pass']['ranked_senses'])  # no exam sense
assert not any('stone' in s['gloss'].lower() for s in by_lemma_c['rock']['ranked_senses'])  # stone not top-4
print('PROBE ASSERTS PASS')

=== rock (tier intermediate, pick_source judge)
  top-1: rock#1 | To play, perform, or enjoy rock music, especially with a lot of skill  | v14c-topic: Society & Culture
  picks: beginner=['rock#23', 'rock#4'] intermediate=['rock#23', 'rock#1', 'rock#4'] advanced=['rock#1', 'rock#23', 'rock#39', 'rock#4']
  top-1 topic trajectory: v14a:Other / Abstract -> v14c:Society & Culture -> v16:Arts & Culture -> v16b:Arts & Culture
  v16b vector: [('Arts & Culture', 1.0)]
=== light (tier beginner, pick_source deterministic)
  top-1: light#62 | any device serving as a source of illumination | v14c-topic: Other / Abstract
  picks: beginner=['light#62', 'light#7'] intermediate=['light#62', 'light#7', 'light#21'] advanced=['light#62', 'light#7', 'light#21', 'light#42']
  top-1 topic trajectory: v14a:Other / Abstract -> v14c:Other / Abstract -> v16:Science & Technology -> v16b:Science & Technology
  v16b vector: [('Science & Technology', 1.0)]
=== pass (tier beginner, pick_source judge)
  top-1: pass#

## ۴) ادغام + داور + مسیر موضوع

- ادغام: ۲۱۰۱ ← ۱۶۷۶ کارت (۴۲۵ حذف). عدد ثبت‌شدهٔ قفل‌شده «۳۶۷ ادغام» است؛ سلول بعد هم‌زمان بازمانده‌های دارای `merged_from` و پیوندهای جذب‌شده را از فایل می‌شمارد و خط evidence را عیناً چاپ می‌کند تا تفاوت تعریف پنهان نماند.
- داور: ۴۹۴ لم با judge و ۶ لم قطعی (light/chairman/cycling/exchange/stool/drive).
- مسیر Other: ‎۲۷٫۰ ← ۳۲٫۹ ← ۱۷٫۷ و چندموضوعی: ۰ ← ۶٫۰ ← ۶٫۹ ← ۱۸٫۷ — همه assert می‌شوند.

In [4]:
removed = sum(len(e['ranked_senses']) for e in ranked_a) - sum(len(e['ranked_senses']) for e in ranked_b)
survivors = [(s['sense_id'], s['merged_from']) for e in ranked_b for s in e['ranked_senses'] if s.get('merged_from')]
links = sum(len(m) for _, m in survivors)
print(f'removed cards: {removed} | survivors carrying merged_from: {len(survivors)} | absorbed links: {links}')
gap_c = (FAC / 'fixtures/gap_report_free_500_v14c.md').read_text(encoding='utf-8')
line367 = [ln for ln in gap_c.splitlines() if '367' in ln][0]
print('locked record from file:', line367.strip())
judged = sum(1 for e in ranked_c if e.get('pick_source') == 'judge')
det = [e['lemma'] for e in ranked_c if e.get('pick_source') != 'judge']
print(f'judged: {judged} | deterministic: {len(det)} {sorted(det)}')
assert judged == 494 and sorted(det) == ['chairman', 'cycling', 'drive', 'exchange', 'light', 'stool']
def other_of(lab): return sum(1 for r in lab if 'Other' in r['topic_label'])
o_c, o_16, o_16b = other_of(lab_c), other_of(lab_16), other_of(lab_16b)
print(f"Other trajectory: {o_c}/1676={100*o_c/1676:.1f}% -> {o_16}/1676={100*o_16/1676:.1f}% -> {o_16b}/1676={100*o_16b/1676:.1f}%")
def multi(vec):
    vs = [v for e in vec for v in e['vectors']]
    m = [v for v in vs if len(v['vector']) > 1]
    bad = [v for v in vs if abs(sum(x['weight'] for x in v['vector']) - 1.0) > 0.011]
    return len(vs), len(m), collections.Counter(len(v['vector']) for v in vs), len(bad)
for name, vv in [('v15', vec_15), ('v16', vec_16), ('v16b', vec_16b)]:
    n, m, shape, bad = multi(vv)
    print(f'{name}: senses={n} multi={m} ({100*m/n:.1f}%) shape={dict(shape)} weight-violations={bad}')
    assert n == 1676 and bad == 0
assert (o_c, o_16, o_16b) == (453, 552, 296)
assert multi(vec_15)[1] == 101 and multi(vec_16)[1] == 116 and multi(vec_16b)[1] == 314
assert other_of(lab_16b) == 296 and multi(vec_16b)[1] == 314
print('MERGE/JUDGE/TOPIC ASSERTS PASS: judged 494/6, Other 453->552->296, multi 0->101->116->314')

removed cards: 425 | survivors carrying merged_from: 374 | absorbed links: 437
locked record from file: زنجیره: v14a قطعی (2101 کارت) ← merge مدل (v14b: 1676 کارت، 367 ادغام، خرابی ۰) ← داور per-level + موضوع (v14c).
judged: 494 | deterministic: 6 ['chairman', 'cycling', 'drive', 'exchange', 'light', 'stool']
Other trajectory: 453/1676=27.0% -> 552/1676=32.9% -> 296/1676=17.7%
v15: senses=1676 multi=101 (6.0%) shape={1: 1575, 2: 99, 3: 2} weight-violations=0
v16: senses=1676 multi=116 (6.9%) shape={1: 1560, 2: 115, 3: 1} weight-violations=0
v16b: senses=1676 multi=314 (18.7%) shape={1: 1362, 2: 312, 3: 2} weight-violations=0
MERGE/JUDGE/TOPIC ASSERTS PASS: judged 494/6, Other 453->552->296, multi 0->101->116->314


## ۵) خلاصهٔ اثبات رجیستری

رجیستری SQLite (`factory/registry.db`) تنها مالک شناسه‌هاست؛ مهاجرت v14 فقط‌خواندنی و بدون پردازش مجدد بود. انتظار قفل‌شده: ۴۹۹ لم (فعل cast در B2 و C1 یک کلید مشترک دارد)، ۱۶۷۱ کارت فعال، و ۴/۴ تست اثبات سبز. سلول بعد مستقیم از دیتابیس (read-only) می‌خواند و خط‌های PASS فایل `REGISTRY_EVIDENCE.md` را استخراج می‌کند.

In [5]:
con = sqlite3.connect(f'file:{(FAC / "registry.db").as_posix()}?mode=ro', uri=True)
cur = con.cursor()
n_lem = cur.execute("select count(*) from lemmas where lang='en'").fetchone()[0]
status = dict(cur.execute("select status, count(*) from precards where lang='en' group by status").fetchall())
dupes = cur.execute('select count(*) from (select pre_card_id from precards where lang=%s group by lang, pre_card_id having count(*) > 1)' % "'en'").fetchone()[0]
print(f"registry en: lemmas={n_lem} active={status.get('active')} superseded={status.get('superseded')} converted={status.get('converted', 0)} dupe-groups={dupes}")
assert n_lem == 499 and status.get('active') == 1671 and dupes == 0
ev = (FAC / 'REGISTRY_EVIDENCE.md').read_text(encoding='utf-8')
passes = [ln.strip() for ln in ev.splitlines() if ln.strip().startswith('- (')] 
print(f'proof tests citing PASS in REGISTRY_EVIDENCE.md: {len(passes)}')
for ln in passes: print('  ', ln[:100])
assert len(passes) == 4
con.close()
print('REGISTRY ASSERTS PASS: 499 lemmas, 1671 active, 4/4 tests')

registry en: lemmas=499 active=1671 superseded=427 converted=0 dupe-groups=0
proof tests citing PASS in REGISTRY_EVIDENCE.md: 4
   - (a) import twice → stats byte-identical, dupe query empty. PASS
   - (b) 20 shuffled duplicate lemmas re-imported → 0 new rows, stats unchanged. PASS
   - (c) `de` + 3 fake lemmas → de=(3/3/3), en untouched; fingerprint
   - (d) `FINAL-en-20260903-01` claim(10) → complete 3 → converted=3;
REGISTRY ASSERTS PASS: 499 lemmas, 1671 active, 4/4 tests


## ۶) دفتر هزینه + رأی صادقانه

کل زنجیره: ۲۹۳ تماس، همه `muse-spark-1.3-contributor-free`، زنجیرهٔ fallback حتی یک بار لازم نشد. اصلاحیه‌های انتقال (شناسهٔ مدل بدون پیشوند `opencode/` وگرنه 401، هدر UA مرورگر وگرنه 403) در CONTEXT ثبت است. سلول بعد بچ‌ها را از progressها و تعداد تماس‌ها را از evidenceها استخراج و جمع را assert می‌کند.

In [6]:
v15ev = (FAC / 'v15_EVIDENCE.md').read_text(encoding='utf-8')
v16ev = (FAC / 'v16_EVIDENCE.md').read_text(encoding='utf-8')
v16bev = (FAC / 'V16B_EVIDENCE.md').read_text(encoding='utf-8')
c15 = int(re.search(r'contributor-free: (\d+)', v15ev).group(1))
c16 = int(re.search(r'Calls: \*\*(\d+)', v16ev).group(1))
c16b = int(re.search(r'Calls: \*\*(\d+)', v16bev).group(1))
ledger = [('v14b merge', 63), ('v14c judge', 63), ('v15 vectors', c15), ('v16 relabel', c16), ('v16b topup', c16b)]
total = sum(c for _, c in ledger)
for name, c0 in ledger: print(f'{name:14} {c0:>3} calls (muse-spark-1.3-contributor-free)')
print('TOTAL:', total)
for name, ev in [('v15', v15ev), ('v16', v16ev), ('v16b', v16bev)]:
    fb = [ln.strip() for ln in ev.splitlines() if 'allback' in ln]
    print(f'{name} fallback lines:', fb[:2])
assert (c15, c16, c16b) == (64, 64, 39)
assert total == 293
assert all('muse-spark-1.3-contributor-free' in ev for ev in [v15ev, v16ev, v16bev])
print('COST ASSERTS PASS: 293 calls, all spark-1.3, zero fallbacks')

v14b merge      63 calls (muse-spark-1.3-contributor-free)
v14c judge      63 calls (muse-spark-1.3-contributor-free)
v15 vectors     64 calls (muse-spark-1.3-contributor-free)
v16 relabel     64 calls (muse-spark-1.3-contributor-free)
v16b topup      39 calls (muse-spark-1.3-contributor-free)
TOTAL: 293
v15 fallback lines: ['- Full run: 63 batches, 0 failed lemmas, 0 fallbacks.', '- Calls per model: **muse-spark-1.3-contributor-free: 64** (1 first-smoke + 1 probe + 62 full; 1 full batch already done by probe smoke). Fallback chain never needed.']
v16 fallback lines: ['- Calls: **64, ALL `muse-spark-1.3-contributor-free`** (chain fallbacks never needed).', '- Failed lemmas: **0** (no deterministic fallback used; all sources `llm-v16`).']
v16b fallback lines: ['Fallback chain never needed. Sleep 2.5s, live tqdm, resume `factory/v16b_progress.json`', '- Failed lemmas: **0** (no keep-Other fallback used; all relabeled rows `llm-v16b`).']
COST ASSERTS PASS: 293 calls, all spark-1.3, zero f

## رأی صادقانه (از همین خروجی‌ها)

- **stone در top-4 نیست**: حس سنگ (stone) حتی در ۴ حس رتبه‌بندی‌شدهٔ rock هم راه نیافت — وزن خالص سقف دارد (درس ثبت‌شده: نگهبان + بوست).
- **حس «امتحان» غایب است**: هیچ‌یک از حس‌های pass معنای امتحان ندارند (فقط v9 تزریقی داشت)؛ نزدیک‌ترین‌ها #۵۵ «تلاش» و #۴۸ «مقطع دشوار» همچنان Other واقعی‌اند.
- **pass#8 کش‌آمده (stretch)**: «cause to pass» با خوانش حرکتی Travel ‏۱٫۰ گرفت —明らًَا force-fit همین اجراست و در evidence پرچم خورده.
- **flat#20/#40 محافظه‌کارانه Other ماندند** (سطح تخت)؛ خوانش Nature/Daily قابل بحث است ولی مدل صادقانه عقب نشست.
- **هدف ≤۱۵٪ از دست رفت**: ‏۱۷٫۷٪ یعنی ۲۹ ردیف تا stretch — ولی ≤۲۰٪ اصلی (HIT) با ۲۵۶ نجات‌یافته از ۵۵۲ محقق شد.